In [97]:
# %% Minimal setup from class

import os, json, textwrap, re, time
import requests

API_KEY  = os.getenv("OPENAI_API_KEY", "cse476")
API_BASE = os.getenv("API_BASE", "http://10.4.58.53:41701/v1")  
MODEL    = os.getenv("MODEL_NAME", "bens_model")              

SYSTEM_PROMPT = "You are a helpful assistant with limited text output. Reply with only the final answer—no explanation. There is no need to repeat the question."
TEMPERATURE   = 0.25 #Must be a float

def call_model_chat_completions(prompt: str,
                                system: str = SYSTEM_PROMPT,
                                model: str = MODEL,
                                temperature: float = TEMPERATURE,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 350,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}

def self_evaluate(question, prediction, expected_answer, model=MODEL):
    """
    Use the model itself as a strict grader.
    Returns True if the model says the prediction matches the expected answer; else False.
    Falls back to a simple normalized string compare if the model's reply is malformed.
    """
    import re

    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."
    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise, return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False
"""

    r = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (r.get("text") or "").strip().lower()
    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: simple normalization-based equality
    norm = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return norm(prediction) == norm(expected_answer)

def self_evaluate_tests(tests, model=MODEL, grader_model=None, sleep_sec=0.2, verbose=True):
    """
    Run the tests by querying the model for each prompt, then use LLM-as-a-judge
    (self_evaluate) to determine correctness.

    Args:
        tests: list of dicts with keys: id, prompt, expected (and optionally type)
        model: model used to generate predictions
        grader_model: model used to judge correctness (defaults to `model` if None)
        sleep_sec: small delay between calls to be polite to the API
        verbose: if True, print a summary line per test

    Returns:
        rows: list of dicts with fields:
              id, expected, got, correct, status, error
    """
    import time

    judge_model = grader_model or model
    rows = []

    for t in tests:
        #1) Get model prediction
        if t.get("input"):
            r = call_model_chat_completions(
                t["input"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )
            got = agent_loop(t["input"])
            #got = reasoning_via_planning(t["input"])
            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["input"],
                prediction=got,
                expected_answer=t["output"],
                model=judge_model,
            )
        else:
            r = call_model_chat_completions(
                t["prompt"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )        #got = (r.get("text") or "").strip()
            got = agent_loop(t["prompt"])

            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["prompt"],
                prediction=got,
                expected_answer=t["expected"],
                model=judge_model,
            )



        row = {
            "id": t.get("id", "<unnamed>"),
            "output": t["output"],
            "got": got,
            "correct": bool(is_correct),
            "status": r.get("status"),
            "error": r.get("error"),
        }
        rows.append(row)

        if verbose:
            mark = "✅" if is_correct else "❌"
            print(f"{mark} {row['id']}: output={row['output']!r}, got={row['got']!r} (HTTP {row['status']})")
            if row["error"]:
                print("   error:", row["error"])

        if sleep_sec:
            time.sleep(sleep_sec)

    return rows


In [98]:
def reasoning_via_planning(prior:str, question: str,) -> dict:
    prior_reasoning = "\nPrior Reasoning: " + prior + "\n\n"
    reasoning_str = "Create a plan using as little words as possible. Using prior reasoning, decompose the problem into a few step to solve the question provided. Execute each step in order to arrive at the final answer.If the question involves math, write a python program that solves the question and exports an answer. Make sure your answer solves the question provided.\n\n Question: "

    r = call_model_chat_completions(
            reasoning_str + question + prior_reasoning,
            system="You are a planner. Provide a short, structured step-by-step plan to solve the question.",
            model=MODEL,
            temperature=0.35,
        )
    got = (r.get("text") or "").strip()
    return got

In [99]:
def tree_of_thought(question: str, n_paths: int, prior: str = None,):
    tot_str = "You will decompose the problem down into {n_paths} distinct possible solution paths to solve the question provided. Depending on the problem, write a short reasonable path that is different from every other path created but still leads to the answer. Expected output should be in the form of: path1:<>, \npath2:<>,etc.\n\n Question: "
    r = call_model_chat_completions(
            tot_str.format(n_paths=n_paths) + question,
            system="You are a problem solver that creates multiple distinct solution paths to solve a problem. Use as little words as possible.",
            model=MODEL,
            temperature=TEMPERATURE,
        )
    #Divide into n_paths
    raw = r.get("text") or ""
    thoughts = []
    for n in range(1, n_paths + 1):
        path = raw.find(f"path{n}:")
        end = raw.find(f"path{n+1}:")
        if end == -1:
            end = len(raw)
        thoughts.append(raw[path:end].strip())
    return thoughts

In [100]:
def self_consistency(question: str, n_paths: int, prior: str = "",):
    paths = []
    for n in range(1, n_paths + 1):
        r = call_model_chat_completions(
                prior + question,
                system="",
                model=MODEL,
                temperature=TEMPERATURE,
            )
        got = (r.get("text") or "").strip()
        paths.append(got)
    return paths

In [101]:
def double_check(prior:str, question: str):
    original_question = "Original Question: " + question + "\n"
    check_str = "You are a careful solver. Use the solution to answer the question. If you find any mistakes, correct them and provide the accurate final answer. If everything is correct, simply restate the correct answer.\n\n Solution to double-check: "
    r = call_model_chat_completions(
            original_question + check_str + prior,
            system="You are a intelligent grader/judge whose job is to validate whether solutions are correct. Given the question and solution, if correct: reiterate the exact answer without paraphrasing. If incorrect provide the corrected answer in correct code, numeric, or textual format. If the answer is a numeric, provide the numeric answer only.",
            model=MODEL,
            temperature=TEMPERATURE,
        )
    got = (r.get("text") or "").strip()
    return got

In [102]:
def critic(prior:str, question: str):
    original_question = "Original Question: " + question + "\n\n"
    crit_str = "Analyze the following solution(s) and choose the single best final answer. Do NOT include explanations — output only the final concise answer.\n\Solution(s):\n"
    r = call_model_chat_completions(
            original_question + crit_str + prior,
            system="You are a meticulous critic. Given solution summaries, select the best final answer without any explanations. If the answer is numeric, provide the numeric answer only.",
            model=MODEL,
            temperature=TEMPERATURE,
        )
    got = (r.get("text") or "").strip()
    return got

<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
C:\Users\isami\AppData\Local\Temp\ipykernel_22364\3824053218.py:3: SyntaxWarning: invalid escape sequence '\S'
  crit_str = "Analyze the following solution(s) and choose the single best final answer. Do NOT include explanations — output only the final concise answer.\n\Solution(s):\n"


In [103]:
import requests, trafilatura
from urllib.parse import quote
# Returns text content of most relevant Wikipedia page for a question
def wiki_tool(question: str):
    p = '''Given a question, provide the title of the most relevant Wikipedia page that answers the question. 
            Only provide the title, no explanations.
            Examples: 
            Question: Which genus of moth in the world's seventh-largest country contains only one species?
            Answer: Crambidae
            Question: What U.S Highway gives access to Zilpo Road, and is also known as Midland Trail?
            Answer: US 60
            \n\n Question: '''
    r = call_model_chat_completions(
            p + question,
            system="You are a Wikipedia search assistant. Given a question, return the title of the most relevant Wikipedia page that answers the question. Reply with only the title, no explanations.",
            model=MODEL,
            temperature=0.3,
        )
    title = (r.get("text") or "").strip()
    
    try:
        #Calls a wiki API to get page of a topic
        url = f"https://en.wikipedia.org/api/rest_v1/page/mobile-html/{quote(title)}"
        headers = {"User-Agent": "MilkBot/1.0 (https://github.com/Isaiah-Milkey)", "Accept": "text/html"}
        r = requests.get(url, headers=headers, timeout=20)
        r.raise_for_status()

        extracted = trafilatura.extract(
            r.text,
            include_tables=True,
            include_comments=False,
            output_format="txt"  
        )
    except Exception as e:
        print(f"Error fetching Wikipedia page for title '{title}': {e}")
        extracted = None
    return extracted or ""

In [ ]:
# Tree of thought (X of thought)
# Reasoning via planning
# 'Wait' am i correct? Double check
# Critic 
# Send to output

# Future:
# Implement RAG or memory of some kind to grab from text data
# implement In-context learning with examples via RAG
# Call to Wikipedia API/ disctionary API/ Calc?

# Agent needs to decide whether to use tools or not
# Make an LLM call to decide wiki tool call if needed
#   Break down into math, topic, general knowledge, planning 

def agent_loop(input_question: str):
    #Have the model decide on a strategy to solve the problem
    r = call_model_chat_completions(
            "Decide what type of category this question falls into: math, coding, general_knowledge, reasoning, or other. Respond with only the category name." + "\n\n Question: " + input_question,
            system="You are a helpful assistant that classifies questions into categories.",
            model=MODEL,
            temperature=0,
        )
    case = (r.get("text") or "").strip().lower()
    case = case.split()[0] if case.split() else "other"
    print(f"Case decided: {case}\n")
    if case in ["math", "coding"]:
        #Math/Coding solving strategy
        print("Using math/coding solving strategy\n")
        curr = reasoning_via_planning("For solving math problems, I can create a simple python program that solves the question- and outputs the answer.", input_question)
        return double_check(curr, input_question)
    elif case == "general_knowledge":
        #Use wiki tool
        print("Using general knowledge solving strategy with wiki tool\n")
        curr = wiki_tool(question=input_question)
        return critic(curr, input_question)
    elif case == "reasoning":
        #Use reasoning via planning
        print("Using reasoning via planning solving strategy\n")
        curr = reasoning_via_planning("No prior reasoning", input_question)
        curr = critic(curr, input_question)
        return double_check(curr, input_question)
    elif case == "other":
        #Fallback to self consistency
        print("Using other case: self consistency solving strategy\n")
        paths = self_consistency(input_question, n_paths=3)
        combined = "These are the different solution paths: \n"
        for p in paths:
            combined += p + "\n"
        return critic(combined, input_question)
    else:
        print("Did not catch a case: Using self consistency\n")
        paths = self_consistency(input_question, n_paths=3)
        combined = "These are the different solution paths: \n"
        for p in paths:
            combined += p + "\n"
        return critic(combined, input_question)
    

In [105]:
import json
import random

with open("cse476_final_project_dev_data.json", "r") as tests:
    DEV_DATA = json.load(tests)

#Get test batches by domain/random
def filter_domain(domain: str):
    filtered = []
    for test in DEV_DATA:
        if test.get("domain") == domain:
            filtered.append(test)
    return filtered

def get_batch(num: int, domain: str = None, is_random: bool = False):
    random.seed(315)
    if domain:
        data = filter_domain(domain)
    else:
        data = DEV_DATA

    if is_random:
        return random.sample(data, num)
    else:
        return data[:num]

In [ ]:

# Example:
# test = reasoning_via_planning("", question="A farmer has 17 sheep and all but 9 are lost. How many sheep are left on the farm?")

# print(test)
#print(test2)

tests = get_batch(5, domain="general_knowledge", is_random=True)
self_evaluate_tests(tests, model=MODEL, sleep_sec=0.5, verbose=True)

# input_question = "What is the capital of France?"
# r = call_model_chat_completions(
#         "Decide what type of category this question falls into: math, coding, general knowledge, reasoning, or other. Respond with only the category name." + "\n\n Question: " + input_question,
#         system="You are a helpful assistant that classifies questions into categories.",
#         model=MODEL,
#         temperature=0.1,
#     )
# case = (r.get("text") or "").strip().lower()
# case = re.split(r'\s+', case)[0]
# print(f"Case decided: {case}\n")


Case decided: coding

Using other case: self consistency solving strategy

❌ <unnamed>: output="    emp_salaries = []\n\n    for prefix, num_employees in dict1.items():\n        if not prefix.startswith('EMPXX'):\n            continue\n\n        for _ in range(num_employees):\n            salary = random.randint(*SALARY_RANGE)\n            emp_salaries.append(salary)\n\n    plt.hist(emp_salaries, bins=10, alpha=0.5)\n    plt.title('Salary Distribution in EMPXX Department')\n    plt.xlabel('Salary')\n    plt.ylabel('Number of Employees')\n    return plt.gca()", got='The best solution is the one that correctly extracts the number of employees from the dictionary, generates random salaries within the specified range, creates a histogram with the correct title and labels, and returns the Axes object. The correct answer is:\n\n```python\nimport random\nimport matplotlib.pyplot as plt\n\n# Constants\nSALARY_RANGE = (20000, 100000)\n\ndef task_func(dict1):\n    # Extract the number of employe

[{'id': '<unnamed>',
  'output': "    emp_salaries = []\n\n    for prefix, num_employees in dict1.items():\n        if not prefix.startswith('EMPXX'):\n            continue\n\n        for _ in range(num_employees):\n            salary = random.randint(*SALARY_RANGE)\n            emp_salaries.append(salary)\n\n    plt.hist(emp_salaries, bins=10, alpha=0.5)\n    plt.title('Salary Distribution in EMPXX Department')\n    plt.xlabel('Salary')\n    plt.ylabel('Number of Employees')\n    return plt.gca()",
  'got': 'The best solution is the one that correctly extracts the number of employees from the dictionary, generates random salaries within the specified range, creates a histogram with the correct title and labels, and returns the Axes object. The correct answer is:\n\n```python\nimport random\nimport matplotlib.pyplot as plt\n\n# Constants\nSALARY_RANGE = (20000, 100000)\n\ndef task_func(dict1):\n    # Extract the number of employees from the department dictionary\n    num_employees = di